In [18]:
import os

In [19]:
!git pull --rebase

Already up to date.


In [20]:
def load_data(dir_location):
    file_list = os.listdir(dir_location)

    for i, file in enumerate(file_list):
        file_list[i] = os.path.join(dir_location, file)

    return file_list

In [21]:
os.getcwd()

'/content/SpaceY'

In [22]:
Voyager1_Files = os.path.join(os.getcwd(),'RAG data','Voyager 1')
Voyager2_Files = os.path.join(os.getcwd(),'RAG data','Voyager 2')
Falcon9_Files = os.path.join(os.getcwd(),'RAG data','Falcon 9')
Starship_Files = os.path.join(os.getcwd(),'RAG data','Starship')

Voyager1_Data = load_data(Voyager1_Files)
Voyager2_Data = load_data(Voyager2_Files)
Falcon9_Data = load_data(Falcon9_Files)
Starship_Data = load_data(Starship_Files)

In [23]:
def File_open(files):

    documents = []
    for file in files:
        with open(file, "r", encoding="utf-8") as f:
            documents.append(f.read())

    return documents

In [27]:
Voyager1 = File_open(Voyager1_Data)
Voyager2 = File_open(Voyager2_Data)
Falcon9 = File_open(Falcon9_Data)
Starship = File_open(Starship_Data)

In [28]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

all_docs = []

for doc in Voyager1:
    all_docs.append({
        "mission": "Voyager1",
        "text": doc
    })

for doc in Voyager2:
    all_docs.append({
        "mission": "Voyager2",
        "text": doc
    })

for doc in Falcon9:
    all_docs.append({
        "mission": "Falcon9",
        "text": doc
    })

for doc in Starship:
    all_docs.append({
        "mission": "Starship",
        "text": doc
    })

print(all_docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[{'mission': 'Voyager1', 'text': "Entering Interstellar Space and Active Science Instruments\n\nVoyager 1 has traveled farther from Earth than any other human-made object. On February 17, 1998, at a distance of 69.4 astronomical units from the Sun, it overtook NASA's Pioneer 10 spacecraft to claim this record. As it continued its outward journey, the probe crossed the termination shock—the boundary where the solar wind abruptly slows down—on December 16, 2004, and entered the outer layer of the heliosphere known as the heliosheath. On August 25, 2012, Voyager 1 crossed the heliopause, leaving the Sun's magnetic influence behind and officially entering interstellar space.\n\nEven in this remote region, Voyager 1 continues to communicate with Earth via NASA's Deep Space Network. The spacecraft measures the properties of interstellar space using its remaining active instruments. These instruments include the Cosmic Ray Subsystem, which monitors high-energy galactic particles; the Low-Ener

In [29]:
print(len(Voyager1))
print(len(Voyager2))
print(len(Falcon9))
print(len(Starship))
print(len(all_docs))

20
20
20
20
80


In [30]:
query = "When was Falcon9 launched?"
query_embedding = model.encode(
    query
)

In [31]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

scores= cosine_similarity(
    [query_embedding],
    [model.encode(doc["text"]) for doc in all_docs]
)

top_k_indices = np.argsort(scores[0])[-3:][::-1]

print(top_k_indices)
for idx in top_k_indices:
    print("=" * 50)
    print("Score:", scores[0, idx])
    print("Mission:", all_docs[idx]["mission"])
    print(all_docs[idx]["text"][:300])

[45 58 52]
Score: 0.6736691
Mission: Falcon9
Development History of Falcon 9

SpaceX began development of the Falcon 9 in the mid-2000s, building upon the lessons learned from its first orbital rocket, the light-lift Falcon 1. The company initially conceived a mid-sized Falcon 5 launcher, but decided to bypass it in favor of the much larger Fa
Score: 0.6074894
Mission: Falcon9
What is Falcon 9?

The Falcon 9 is a two-stage-to-orbit launch vehicle designed, manufactured, and operated by SpaceX. First launched on June 4, 2010, the rocket was developed to provide reliable and safe transport of satellites and cargo, and eventually humans, into Earth orbit. One of its most def
Score: 0.48860425
Mission: Falcon9
Evolution of Falcon 9 Versions

Since its debut in 2010, the Falcon 9 has undergone significant design changes to increase its payload capacity, reliability, and reusability. The initial version, Falcon 9 v1.0, featured a 3x3 engine grid and lower thrust Merlin 1C engines. In 2013, S

In [33]:
context= ''

for idx in top_k_indices:
  context += all_docs[idx]['text']
  context += '\n\n'

print(context)

Development History of Falcon 9

SpaceX began development of the Falcon 9 in the mid-2000s, building upon the lessons learned from its first orbital rocket, the light-lift Falcon 1. The company initially conceived a mid-sized Falcon 5 launcher, but decided to bypass it in favor of the much larger Falcon 9 to meet the payload demands of commercial satellite operators and NASA's ISS resupply program.

The rocket's development was funded in part by NASA's Commercial Orbital Transportation Services (COTS) program, which sought to establish commercial alternatives for ISS supply runs. The first flight of Falcon 9, version 1.0, took place on June 4, 2010, from Space Launch Complex 40 at Cape Canaveral. The flight was a success, placing a dummy test payload into orbit and validating the fundamental design, including the multi-engine first stage and the Merlin engine design.


What is Falcon 9?

The Falcon 9 is a two-stage-to-orbit launch vehicle designed, manufactured, and operated by SpaceX.

In [34]:
prompt = f"""
You are a Space Mission Assistant.

Your task is to answer questions ONLY using the provided context.

Rules:
1. Use only information explicitly present in the context.
2. Do NOT use your own knowledge.
3. If the answer cannot be found in the context, reply exactly:
   "I could not find enough information in the provided documents."
4. Do not make assumptions or guesses.
5. If multiple context documents contain relevant information, combine them into a concise answer.
6. After the answer, provide the sources used.

Context:
{context}

Question:
{query}

Answer:
"""

In [ ]:
from dotenv import load_env
OPENROUTER_API = os.environ.get("OPENROUTER_API_KEY", "")

import requests
import json

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {OPENROUTER_API}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "poolside/laguna-xs.2:free",
    "messages": messages,
    "reasoning": {"enabled": True}
  })
)

if response.status_code == 200:
    response_data = response.json()
    if 'choices' in response_data and len(response_data['choices']) > 0:
        assistant_message = response_data['choices'][0]['message']
        final_answer = assistant_message.get('content')
        print(final_answer)
    else:
        print("Error: No choices found in the response.")
else:
    print(f"Error: OpenRouter API call failed with status code {response.status_code}")
    print(response.text)



The Falcon 9 was first launched on June 4, 2010.

**Sources used:**
- Development History of Falcon 9: "The first flight of Falcon 9, version 1.0, took place on June 4, 2010, from Space Launch Complex 40 at Cape Canaveral."
- What is Falcon 9: "First launched on June 4, 2010"



In [36]:
import pickle

all_embeddings = []
for doc_item in all_docs:
    all_embeddings.append(model.encode(doc_item["text"]))

data = {
    "docs": all_docs,
    "embeddings": all_embeddings,
}

with open("RAG_model.pkl", "wb") as f:
    pickle.dump(data, f)